# AHFL Current Version — Drive-based current-code smoke test

Tests the current repo code from Google Drive.

Focus:
- current PaddleOCR API
- current repo helper import path
- OCR init via `create_paddle_ocr()`
- optional pipeline smoke test via `process_image()`

This notebook is for current changes only.

In [ ]:
import sys, os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

PROJECT = '/content/drive/MyDrive/AHFL-GPU-Tests/ahfl-working-Gpu'
IMAGE_DIR = os.path.join(PROJECT, 'imges')
OUTPUT_DIR = os.path.join(PROJECT, 'colab_test_outputs')

if not os.path.exists(PROJECT):
    raise FileNotFoundError(f'Project not found: {PROJECT}')

os.makedirs(OUTPUT_DIR, exist_ok=True)
sys.path.insert(0, PROJECT)

print(f'✓ Project: {PROJECT}')
print(f'✓ Image dir: {IMAGE_DIR}')
print(f'✓ Output dir: {OUTPUT_DIR}')

In [ ]:
import subprocess, sys

packages = [
    'paddleocr==3.4.0',
    'opencv-python-headless',
    'Pillow',
    'numpy',
    'python-dotenv',
    'pydantic==2.7.0',
    'pytesseract',
    'ultralytics==8.3.235',
    'img2pdf',
    'pdf2image',
    'PyPDF2',
]

try:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'paddlepaddle-gpu==3.3.0',
        '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu121/'
    ])
    print('✓ paddlepaddle-gpu 3.3.0')
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'paddlepaddle==3.3.0'])
    print('⚠ CPU fallback: paddlepaddle 3.3.0')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

import platform, paddle, paddleocr, torch, cv2
print('Python      :', platform.python_version())
print('Paddle      :', paddle.__version__)
print('PaddleOCR   :', paddleocr.__version__)
print('Torch       :', torch.__version__)
print('CUDA avail  :', torch.cuda.is_available())
print('OpenCV      :', cv2.__version__)

In [ ]:
from core.ocr.paddle import create_paddle_ocr
from core.pipeline import process_image

ocr = create_paddle_ocr()
print('✓ OCR ready via repo helper:', type(ocr).__name__)
print('✓ process_image imported')

In [ ]:
from glob import glob
import os

patterns = ['*.png', '*.jpg', '*.jpeg', '*.PDF', '*.pdf']
images = []
for pattern in patterns:
    images.extend(sorted(glob(os.path.join(IMAGE_DIR, pattern))))

images = sorted(set(images))
print(f'Found {len(images)} test files in Drive image dir')
if not images:
    from google.colab import files
    print('No Drive images found. Upload one or more files now:')
    uploaded = files.upload()
    images = list(uploaded.keys())

results = {}
for img in images[:5]:
    name = os.path.basename(img)
    print(f'\n→ OCR: {name}')
    try:
        result = ocr.ocr(img)
        results[name] = result
        lines = result[0] if result and result[0] else []
        print(f'  regions: {len(lines)}')
        for line in lines[:3]:
            print(f'    "{line[1][0]}" ({line[1][1]:.3f})')
    except Exception as e:
        results[name] = {'error': str(e)}
        print(f'  ✗ {e}')

In [ ]:
import json
import cv2

json_f = os.path.join(OUTPUT_DIR, 'current_version_results.json')
serializable = {}
for name, result in results.items():
    if isinstance(result, dict) and 'error' in result:
        serializable[name] = result
        continue
    lines = result[0] if result and result[0] else []
    serializable[name] = [
        {'text': line[1][0], 'conf': round(float(line[1][1]), 4)}
        for line in lines
    ]

with open(json_f, 'w') as f:
    json.dump(serializable, f, indent=2, ensure_ascii=False)

print(f'✓ Saved OCR summary: {json_f}')

pipeline_status = 'skipped'
if images:
    sample_img = images[0]
    img = cv2.imread(sample_img)
    if img is not None:
        try:
            masked_img, report = process_image(img, skip_keywords_enabled=True)
            out_img = os.path.join(OUTPUT_DIR, f'masked_{os.path.basename(sample_img)}')
            cv2.imwrite(out_img, masked_img)
            pipeline_status = 'ok'
            print(f'✓ Saved masked preview: {out_img}')
            print('Report keys:', sorted(report.keys()))
            print('lane_chosen:', report.get('lane_chosen'))
            print('mask_counts keys:', sorted((report.get('mask_counts') or {}).keys()))
        except Exception as e:
            pipeline_status = f'pipeline_error: {e}'
            print(f'⚠ Pipeline smoke test failed: {e}')
    else:
        pipeline_status = 'pipeline_skipped_non_image_input'

print('Pipeline status:', pipeline_status)
print('\nOutput files:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    print('  •', f)